# PulseGuard AI: Biomedical Signal Classifier
## Multi-Paradigm Arrhythmia Detection & Clinical Telemetry Triage
**AAMI EC57 / PhysioNet MIT-BIH Benchmark Standard (Lead II, 250 Hz)**

This notebook contains the complete, self-contained end-to-end implementation of **PulseGuard AI**:
1. **Deterministic DSP**: Zero-Phase Butterworth Bandpass (0.5–45 Hz) & 50 Hz IIR Notch Filtering
2. **Fiducial Localization**: Pan-Tompkins QRS Complex & R-Peak Detection Engine
3. **Feature Engineering**: Daubechies-4 (`db4`) Discrete Wavelet Transform (DWT) & Temporal HRV Descriptors (36-dim)
4. **Multi-Paradigm AI**: Random Forest, SVM-RBF, Deep 1D-CNN (Temporal ResNet), and a 4-Qubit Hybrid Quantum Neural Network (QNN)
5. **Clinical Decision Support**: 5-Tier AHA/ACC Triage Escalation (`CRITICAL EMERGENCY` to `NORMAL`)

### 1. Setup & Dependencies Installation

In [ ]:
# Install dependencies if running on Google Colab
import sys
!pip install -q pennylane torch pywavelets scikit-learn scipy matplotlib seaborn plotly fastapi uvicorn

In [ ]:
import numpy as np
import scipy.signal as signal
import pywt
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import pennylane as qml
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, recall_score, f1_score, roc_auc_score, confusion_matrix

print('PyTorch Version :', torch.__version__)
print('PennyLane Version:', qml.__version__)
print('PyWavelets Version:', pywt.__version__)

### 2. Configuration & Physiological Standards
Under ANSI/AAMI EC57, arrhythmias are mapped to five diagnostic categories:
* **Class 0 (N)**: Normal Sinus Rhythm (NSR)
* **Class 1 (S)**: Atrial Fibrillation / Supraventricular Ectopy (AFIB/SVEB)
* **Class 2 (V)**: Premature Ventricular Contractions (PVC/VEB)
* **Class 3 (B)**: Left Bundle Branch Block (LBBB)
* **Class 4 (E)**: Myocardial Ischemia / ST-T Anomaly (ST)

In [ ]:
SAMPLING_RATE = 250   # Hz
PRE_R_SAMPLES = int(0.25 * SAMPLING_RATE)   # 250 ms before R-peak (62 samples)
POST_R_SAMPLES = int(0.45 * SAMPLING_RATE)  # 450 ms after R-peak (113 samples)
BEAT_WINDOW_LEN = PRE_R_SAMPLES + POST_R_SAMPLES  # 175 samples (700 ms)

CLASSES = {
    0: 'Normal Sinus Rhythm (NSR)',
    1: 'Atrial Fibrillation / SVEB (AFIB)',
    2: 'Premature Ventricular Contraction (PVC)',
    3: 'Left Bundle Branch Block (LBBB)',
    4: 'Ischemia / ST-T Abnormality (ST)',
}
CLASS_SHORT = {0: 'Normal', 1: 'AFIB', 2: 'PVC', 3: 'LBBB', 4: 'Ischemia'}

### 3. Digital Signal Processing (DSP) & Pan-Tompkins QRS Detector

In [ ]:
def butter_bandpass_filter(data, lowcut=0.5, highcut=45.0, fs=SAMPLING_RATE, order=4):
    nyq = 0.5 * fs
    b, a = signal.butter(order, [lowcut / nyq, highcut / nyq], btype='band')
    return signal.filtfilt(b, a, data)  # Zero-phase forward-backward filtering

def notch_filter(data, freq=50.0, fs=SAMPLING_RATE, Q=30.0):
    b, a = signal.iirnotch(freq, Q, fs)
    return signal.filtfilt(b, a, data)

def clean_ecg_signal(raw, fs=SAMPLING_RATE):
    return notch_filter(butter_bandpass_filter(raw, fs=fs), fs=fs)

def pan_tompkins_detector(ecg_signal, fs=SAMPLING_RATE):
    nyq = 0.5 * fs
    b, a = signal.butter(3, [5.0 / nyq, 15.0 / nyq], btype='band')
    bpass = signal.filtfilt(b, a, ecg_signal)
    
    h_der = np.array([-2, -1, 0, 1, 2]) / 8.0
    der = np.convolve(bpass, h_der, mode='same')
    squared = der ** 2
    
    win_len = int(0.150 * fs)  # 150 ms MWI
    win = np.ones(win_len) / win_len
    integrated = np.convolve(squared, win, mode='same')
    
    min_dist = int(0.200 * fs) # 200 ms refractory blanking
    thresh = 0.35 * np.mean(integrated) + 0.15 * np.max(integrated)
    c_peaks, _ = signal.find_peaks(integrated, distance=min_dist, height=thresh)
    
    search_r = int(0.080 * fs)
    true_r = []
    for cp in c_peaks:
        start, end = max(0, cp - search_r), min(len(ecg_signal), cp + search_r)
        if end > start:
            true_r.append(start + np.argmax(ecg_signal[start:end]))
            
    r_peaks = []
    for p in true_r:
        if not r_peaks or (p - r_peaks[-1]) >= min_dist:
            r_peaks.append(p)
    return np.array(r_peaks, dtype=int), integrated

### 4. Daubechies-4 (`db4`) Discrete Wavelet & HRV Feature Extraction

In [ ]:
def extract_beat_features(beat, prev_rr, post_rr, mean_rr):
    # 4-level db4 DWT decomposition (A4, D4, D3, D2, D1)
    coeffs = pywt.wavedec(beat, 'db4', level=4)
    w_feats = []
    for c in coeffs:
        energy = float(np.sum(c ** 2) / len(c))
        var = float(np.var(c))
        prob = (c ** 2) / (np.sum(c ** 2) + 1e-12)
        entropy = float(-np.sum(prob * np.log(prob + 1e-12)))
        std = float(np.std(c))
        max_amp = float(np.max(np.abs(c)))
        w_feats.extend([energy, var, entropy, std, max_amp])
        
    # Morphological descriptors
    r_amp = float(np.max(beat))
    s_trough = float(np.min(beat))
    p2p = r_amp - s_trough
    sig_std = float(np.std(beat))
    skew = float(np.mean((beat - np.mean(beat))**3) / (sig_std**3 + 1e-8))
    kurt = float(np.mean((beat - np.mean(beat))**4) / (sig_std**4 + 1e-8))
    zc = float(np.sum(np.diff(np.sign(beat)) != 0))
    
    # Temporal HRV descriptors
    rr_ratio = float(prev_rr / (post_rr + 1e-6))
    rr_diff = float(np.abs(prev_rr - mean_rr))
    
    all_f = w_feats + [r_amp, s_trough, p2p, sig_std, skew, kurt, zc, prev_rr, post_rr, rr_ratio, rr_diff]
    return np.array(all_f, dtype=np.float32)

### 5. Benchmark Cohort Synthesis (PhysioNet AAMI Standard)

In [ ]:
from biomed_classifier.data import generate_benchmark_cohort, generate_continuous_rhythm

cohort = generate_benchmark_cohort(n_samples_per_class=120)
print('Dataset Features Shape :', cohort.X_features.shape)  # (600, 36)
print('Raw Waveforms Shape    :', cohort.raw_beats.shape)   # (600, 175)
print('Class Distribution     :', np.bincount(cohort.y_labels))

# Stratified train/test split (80% train, 20% test)
X_tr, X_te, y_tr, y_te = train_test_split(
    cohort.X_features, cohort.y_labels, test_size=0.20, stratify=cohort.y_labels, random_state=42
)
scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s = scaler.transform(X_te)

### 6. Quantum Neural Network: 4-Qubit PQC with Residual Highway

In [ ]:
dev = qml.device('default.qubit', wires=4)

@qml.qnode(dev, interface='torch', diff_method='backprop')
def pqc_circuit(inputs, weights):
    qml.AngleEmbedding(inputs, wires=range(4), rotation='Y')
    qml.StronglyEntanglingLayers(weights, wires=range(4))
    return [qml.expval(qml.PauliZ(i)) for i in range(4)]

class ResidualHybridQNN(nn.Module):
    def __init__(self, n_features=36, n_qubits=4, n_layers=2, n_classes=5):
        super().__init__()
        self.ingress = nn.Sequential(nn.Linear(n_features, n_qubits), nn.Tanh())
        self.qlayer = qml.qnn.TorchLayer(pqc_circuit, {'weights': (n_layers, n_qubits, 3)})
        # Residual Quantum-Classical Highway combining latent angles & quantum expectation values
        self.head = nn.Sequential(
            nn.Linear(n_qubits * 2, 16),
            nn.GELU(),
            nn.Linear(16, n_classes)
        )
    def forward(self, x):
        latent = self.ingress(x)
        q_out = self.qlayer(latent)
        combined = torch.cat([latent, q_out], dim=-1)
        return self.head(combined)

### 7. Multi-Paradigm Training & Benchmarking

In [ ]:
# 1. Classical Models
rf = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42)
rf.fit(X_tr_s, y_tr)
rf_preds = rf.predict(X_te_s)

svm = SVC(kernel='rbf', class_weight='balanced', probability=True, random_state=42)
svm.fit(X_tr_s, y_tr)
svm_preds = svm.predict(X_te_s)

# 2. Quantum Hybrid Model Training
qnn = ResidualHybridQNN()
opt = torch.optim.AdamW(qnn.parameters(), lr=0.02, weight_decay=1e-4)
crit = nn.CrossEntropyLoss()
X_t, y_t = torch.tensor(X_tr_s, dtype=torch.float32), torch.tensor(y_tr, dtype=torch.long)

print('Training Residual Hybrid QNN (35 Epochs)...')
for ep in range(35):
    opt.zero_grad()
    out = qnn(X_t)
    loss = crit(out, y_t)
    loss.backward()
    opt.step()

with torch.no_grad():
    qnn_probs = torch.softmax(qnn(torch.tensor(X_te_s, dtype=torch.float32)), dim=-1).numpy()
    qnn_preds = np.argmax(qnn_probs, axis=-1)

print('Random Forest Accuracy :', accuracy_score(y_te, rf_preds) * 100)
print('SVM-RBF Accuracy       :', accuracy_score(y_te, svm_preds) * 100)
print('Residual QNN Accuracy  :', accuracy_score(y_te, qnn_preds) * 100)

### 8. Clinical Telemetry Case Study: Bed 402 Ventricular Bigeminy

In [ ]:
from biomed_classifier.service import get_service

svc = get_service()
patient = svc.get_benchmark_patient(condition='pvc', duration=10.0)
analysis = svc.analyze_rhythm(patient['raw_signal'], selected_model='Hybrid QNN (4 Qubits + PQC)')

print(f'Detected Beats : {analysis["total_beats"]} beats | Heart Rate: {analysis["heart_rate_bpm"]} BPM')
print(f'Triage Level   : [{analysis["triage"]["level"]}]')
print(f'Recommendation : {analysis["triage"]["recommendation"]}')

# Plot ECG with detected R-Peaks
plt.figure(figsize=(12, 4))
t_ax = patient['time_axis']
plt.plot(t_ax, analysis['filtered_signal'], color='#2980b9', lw=1.2, label='Filtered ECG (Lead II)')
r_p = analysis['r_peaks']
plt.plot(t_ax[r_p], analysis['filtered_signal'][r_p], 'ro', markersize=6, label='Detected R-Peaks')
plt.title('Clinical Telemetry Monitor: Bed 402 Ventricular Bigeminy (Alternating PVC Sequences)')
plt.xlabel('Time (seconds)')
plt.ylabel('Amplitude (mV)')
plt.legend()
plt.show()

### 9. Launch the Web Telemetry Dashboard or REST API
To launch the interactive Streamlit dashboard or FastAPI backend from terminal:
```bash
python main.py --app    # Streamlit Dashboard (http://localhost:8501)
python main.py --api    # FastAPI REST Engine (http://127.0.0.1:8502/docs)
python main.py --test   # Automated 23-test Pytest Suite
```